This notebook provides examples to go along with the [textbook](http://manipulation.csail.mit.edu/robot.html).  I recommend having both windows open, side-by-side!

In [1]:
import sys

if "google.colab" in sys.modules:
    %pip install -q manipulation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from pydrake.all import ModelVisualizer, PackageMap

from manipulation import running_as_notebook
from manipulation.meshcat_utils import StartMeshcat

In [3]:
# Start the visualizer.
meshcat = StartMeshcat()

<IPython.core.display.Javascript object>

Or run DisplayMeshcat(meshcat) to embed it in this notebook.


# Robot arms

The next two cells will give you a simple interface to move the joints around on a variety of robot arm models.  I highly recommend opening the "controls" menu in the meshcat visualizer. You can expand the tree under "drake", then check and uncheck the different properties (illustration / inertia / proximity) to explore the model (and the visualization tool).

Have a favorite robot that I haven't included here?  If you send me a link to a properly licensed URDF or SDF description of the robot and it's mesh files, I'm happy to add it!  It's worth taking a look at the files quickly, to get a sense for how they work: [SDF](https://github.com/RobotLocomotion/models/blob/master/iiwa_description/sdf/iiwa14_no_collision.sdf), [URDF](https://github.com/RobotLocomotion/models/blob/master/iiwa_description/urdf/iiwa14_no_collision.urdf).

In [4]:
# First pick your robot by un-commenting one of these:
robot = "Kuka LBR iiwa 7"
# robot = "Kuka LBR iiwa 14"
# robot = "Kinova Jaco Gen2 (7 DoF)"
# robot = "Franka Emika Panda"
# robot = "UR3e"


def get_model_url(description):
    # Note: I could download remote model resources here if necessary.
    if description == "Kuka LBR iiwa 7":
        return (
            "package://drake_models/iiwa_description/sdf/iiwa7_with_box_collision.sdf"
        )
    elif description == "Kuka LBR iiwa 14":
        return "package://drake_models/iiwa_description/urdf/iiwa14_primitive_collision.urdf"
    elif description == "Kinova Jaco Gen2 (7 DoF)":
        return "package://drake_models/jaco_description/urdf/j2s7s300.urdf"
    elif description == "Franka Emika Panda":
        return "package://drake_models/franka_description/urdf/panda_arm_hand.urdf"
    elif description == "UR3e":
        return (
            "package://drake_models/ur_description/urdf/ur3e_cylinders_collision.urdf"
        )
    raise Exception("Unknown model")


visualizer = ModelVisualizer(meshcat=meshcat)
visualizer.AddModels(url=get_model_url(robot))

visualizer.Run(loop_once=not running_as_notebook)
meshcat.DeleteAddedControls()

INFO:drake:PackageMap: Downloading https://github.com/RobotLocomotion/models/archive/fa93453f12f50855d46ab1a4f94987ac2319a9ea.tar.gz
INFO:drake:Click 'Stop Running' or press Esc to quit


# Robot hands

I don't have the same smörgåsbord of robot models to offer for robot hands (yet).  We do have the allegro hand model available, and I'm happy to add more here (though many will require some care to handle their kinematic constraints).  For now, you can at least try out the [Allegro Hand](https://www.allegrohand.com/).

TODO(russt): Consider porting Robotiq, Sandia, or IHY-hand from our [openhumanoids](https://github.com/openhumanoids/oh-distro/tree/master/software/models/common_components) project.

In [5]:
meshcat.Delete()

visualizer = ModelVisualizer(meshcat=meshcat)
visualizer.AddModels(
    url="package://drake_models/allegro_hand_description/sdf/allegro_hand_description_right.sdf"
)

visualizer.Run(loop_once=not running_as_notebook)

meshcat.DeleteAddedControls()

INFO:drake:Click 'Stop Running' or press Esc to quit


# Robots description files from other repositories

We've tried to make it very easy to pull in resources from other repositories.  Here's an example of pulling in files directly from github.

## DeepMind Control Suite

In [6]:
if running_as_notebook:  # I don't want to download the dm_control suite in CI.
    meshcat.Delete()
    visualizer = ModelVisualizer(meshcat=meshcat)
    visualizer.parser().package_map().AddRemote(
        package_name="dm_control",
        params=PackageMap.RemoteParams(
            urls=[
                f"https://github.com/google-deepmind/dm_control/archive/refs/tags/1.0.15.tar.gz"
            ],
            sha256=("bac091b18689330a99b7c18ddf86baa916527f5e4ab8e3ded0c8caff1dab2048"),
            strip_prefix="dm_control-1.0.15/",
        ),
    )
    visualizer.AddModels(url="package://dm_control/dm_control/suite/cheetah.xml")
    plant = visualizer.parser().plant()
    visualizer.Run(loop_once=not running_as_notebook)
    meshcat.DeleteAddedControls()

INFO:drake:PackageMap: Downloading https://github.com/google-deepmind/dm_control/archive/refs/tags/1.0.15.tar.gz
INFO:drake:Click 'Stop Running' or press Esc to quit


## MuJoCo Menagerie

Another nice repository of robot models is the [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie).  See the [menagerie.ipynb](menagerie.ipynb) notebook to browse these models in Drake+Meshcat.
